# Aula 03 - Notebook: Verificador Algorítmico de Tautologias e Prova Formal de Segurança

Neste notebook implementamos um motor de classificação semântica de fórmulas booleanas capaz de gerar tabelas-verdade completas para $2^n$ estados e comprovar formalmente a integridade das matrizes de intertravamento de segurança.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

import itertools
from typing import Callable, Dict, List, Any

class VerificadorSemantico:
    @staticmethod
    def avaliar_formula(variaveis: List[str], formula_fn: Callable[[Dict[str, bool]], bool]) -> Dict[str, Any]:
        n = len(variaveis)
        total_estados = 2 ** n
        verdadeiras = 0
        falsas = 0
        
        for tupla in itertools.product([False, True], repeat=n):
            env = dict(zip(variaveis, tupla))
            if formula_fn(env):
                verdadeiras += 1
            else:
                falsas += 1
                    
        if verdadeiras == total_estados:
            classificacao = "TAUTOLOGIA (SEMPRE VERDADEIRO)"
        elif falsas == total_estados:
            classificacao = "CONTRADIÇÃO (SEMPRE FALSO / INSATISFATÍVEL)"
        else:
            classificacao = "CONTINGÊNCIA (SATISFATÍVEL)"
            
        return {
            "Total Estados": total_estados,
            "Contagem True": verdadeiras,
            "Contagem False": falsas,
            "Classificação": classificacao
        }

print("[OK] Motor de Verificação Semântica carregado com sucesso!")


[OK] Motor de Verificação Semântica carregado com sucesso!


In [2]:
def prova_seguranca_sobrepressao(env: Dict[str, bool]) -> bool:
    risco = env['p1'] and env['v1']
    intertrava = (not env['p1']) or (not env['v1'])
    return risco and intertrava

def teorema_invariante_seguranca(env: Dict[str, bool]) -> bool:
    return not prova_seguranca_sobrepressao(env)

def intertravamento_reator_fn(env: Dict[str, bool]) -> bool:
    falha = env['p1'] or env['t1'] or env['g1'] or env['e1']
    consequente = (not env['v1']) and (not env['v2']) and env['a1']
    return (not falha) or consequente

testes = [
    ("Prova de Risco sob Intertrava (Estado Proibido)", ['p1', 'v1'], prova_seguranca_sobrepressao),
    ("Teorema Invariante de Segurança (Planta Segura)", ['p1', 'v1'], teorema_invariante_seguranca),
    ("Regra de Intertravamento do Reator", ['p1', 't1', 'g1', 'e1', 'v1', 'v2', 'a1'], intertravamento_reator_fn),
]

relatorio = []
for nome, vars_list, fn in testes:
    res = VerificadorSemantico.avaliar_formula(vars_list, fn)
    relatorio.append({
        "Expressão / Teorema": nome,
        "Qtd Variáveis": len(vars_list),
        "Espaço Estados": res["Total Estados"],
        "True": res["Contagem True"],
        "False": res["Contagem False"],
        "Resultado Semântico": res["Classificação"]
    })

print(formatar_tabela(relatorio))

assert relatorio[0]["Resultado Semântico"] == "CONTRADIÇÃO (SEMPRE FALSO / INSATISFATÍVEL)"
assert relatorio[1]["Resultado Semântico"] == "TAUTOLOGIA (SEMPRE VERDADEIRO)"
print("\n[OK] Todas as provas lógicas e propriedades de segurança funcional validadas com sucesso!")


Expressão / Teorema                             | Qtd Variáveis | Espaço Estados | True | False | Resultado Semântico                        
------------------------------------------------+---------------+----------------+------+-------+--------------------------------------------
Prova de Risco sob Intertrava (Estado Proibido) | 2             | 4              | 0    | 4     | CONTRADIÇÃO (SEMPRE FALSO / INSATISFATÍVEL)
Teorema Invariante de Segurança (Planta Segura) | 2             | 4              | 4    | 0     | TAUTOLOGIA (SEMPRE VERDADEIRO)             
Regra de Intertravamento do Reator              | 7             | 128            | 23   | 105   | CONTINGÊNCIA (SATISFATÍVEL)                

[OK] Todas as provas lógicas e propriedades de segurança funcional validadas com sucesso!
